# Laboratory Task 6: CNN Architecture in PyTorch

<h3 style="font-family: Times New Roman"><strong>Laboratory Task 6</strong></h3>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px"><b>Instruction:</b> Convert the CNN architecture diagram into a PyTorch CNN architecture.</p>

<h4 style="font-family: Times New Roman"><strong>Import Standard Libraries</strong></h4>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

<h4 style="font-family: Times New Roman"><strong>Output Shape Check</strong></h4>

$$\text{Output Shape} = \left\lfloor\frac{\text{input width} - \text{filter} + 2 \cdot \text{padding}}{\text{stride}}\right\rfloor + 1$$

In [2]:
def calc_out(w, f, s, p):
    return (w - f + 2 * p) // s + 1

w = 28
print("Conv1    :", calc_out(w, 3, 1, 1))
w = calc_out(w, 3, 1, 1)
print("MaxPool1 :", calc_out(w, 2, 2, 1))
w = calc_out(w, 2, 2, 1)
for name in ["Conv2", "Conv3", "Conv4"]:
    w = calc_out(w, 3, 1, 1)
    print(f"{name:<9}:", w)
print("MaxPool2 :", calc_out(w, 2, 2, 0))
w = calc_out(w, 2, 2, 0)
print("Flatten  :", 256 * w * w)

Conv1    : 28
MaxPool1 : 15
Conv2    : 15
Conv3    : 15
Conv4    : 15
MaxPool2 : 7
Flatten  : 12544


<h4 style="font-family: Times New Roman"><strong>CNN Architecture</strong></h4>

In [3]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), stride=2, padding=1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=(3, 3), stride=1, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=(3, 3), stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), stride=2, padding=0)

        self.dropout = nn.Dropout(p=0.2)
        self.flatten = nn.Flatten()

        self.fcn1 = nn.Linear(256 * 7 * 7, 1000)
        self.fcn2 = nn.Linear(1000, 500)
        self.fcn3 = nn.Linear(500, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        x = self.dropout(x)
        x = self.flatten(x)

        x = F.relu(self.fcn1(x))
        x = F.relu(self.fcn2(x))
        x = F.softmax(self.fcn3(x), dim=1)

        return x

<h4 style="font-family: Times New Roman"><strong>Instantiate the Class</strong></h4>

In [4]:
model = CNN(num_classes=10)
model

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fcn1): Linear(in_features=12544, out_features=1000, bias=True)
  (fcn2): Linear(in_features=1000, out_features=500, bias=True)
  (fcn3): Linear(in_features=500, out_features=10, bias=True)
)

<h4 style="font-family: Times New Roman"><strong>Verify the Shapes Layer by Layer</strong></h4>

In [5]:
for name, layer in model.named_children():
    layer.register_forward_hook(lambda m, i, o, n=name: print(f"{n:<8} -> {tuple(o.shape)}"))

x = torch.randn(32, 1, 28, 28)
print(f"{'input':<8} -> {tuple(x.shape)}")
out = model(x)

print("\nrow sums of the output:", out.sum(dim=1)[:4])

input    -> (32, 1, 28, 28)


conv1    -> (32, 32, 28, 28)


pool1    -> (32, 32, 15, 15)
conv2    -> (32, 64, 15, 15)
conv3    -> (32, 128, 15, 15)
conv4    -> (32, 256, 15, 15)
pool2    -> (32, 256, 7, 7)
dropout  -> (32, 256, 7, 7)
flatten  -> (32, 12544)


fcn1     -> (32, 1000)
fcn2     -> (32, 500)
fcn3     -> (32, 10)

row sums of the output: 

tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SliceBackward0>)


<h4 style="font-family: Times New Roman"><strong>Model Parameters</strong></h4>

In [6]:
def count_parameters(model):
    params = {n: p.numel() for n, p in model.named_parameters() if p.requires_grad}
    for n, c in params.items():
        print(f"{n:<14}{c:>12,}")
    print("_" * 26)
    print(f"{'total':<14}{sum(params.values()):>12,}")

count_parameters(model)

conv1.weight           288
conv1.bias              32
conv2.weight        18,432
conv2.bias              64
conv3.weight        73,728
conv3.bias             128
conv4.weight       294,912
conv4.bias             256
fcn1.weight     12,544,000
fcn1.bias            1,000
fcn2.weight        500,000
fcn2.bias              500
fcn3.weight          5,000
fcn3.bias               10
__________________________
total           13,438,350


<p style="font-family:Times New Roman; text-align:justify; font-size:15px"><b>Interpretation:</b></p>
<ul style="font-family:Times New Roman; font-size:15px; text-align:justify">
<li>The channels grow 1 → 32 → 64 → 128 → 256 through the four convolutions. Each uses a 3×3 kernel with stride 1 and padding 1, so it keeps the spatial size.</li>
<li>MaxPool1 uses padding=1 as shown in the diagram, so 28×28 becomes 15×15. MaxPool2 has no padding, so 15×15 becomes 7×7. This is why the "?" in the diagram resolves to 15 for Conv2 to Conv4 and 7 after MaxPool2.</li>
<li>Flatten gives 256 × 7 × 7 = 12,544 features, so FCN1 is <tt>Linear(12544, 1000)</tt>, which fills in the "?" for its input.</li>
<li>The output size of FCN3 is also a "?" in the diagram. It equals the number of classes, so it is a constructor argument (<tt>num_classes</tt>), shown here with 10.</li>
<li>The diagram ends with SoftMax, so it is in <tt>forward</tt>. If this model is trained with <tt>nn.CrossEntropyLoss</tt>, which applies log-softmax itself, the softmax should be removed.</li>
</ul>